In [ ]:
# ============================================================
# Hybrid LSTM-GARCH Framework — S&P 500 Volatility Forecasting
# Step 1: Environment setup and data download
# ============================================================

# Install required packages (run once)
# pip install yfinance arch tensorflow scikit-learn pandas numpy matplotlib seaborn

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# ── Download SPY data ─────────────────────────────────────────
print("Downloading SPY data from Yahoo Finance...")

spy = yf.download('SPY',
                  start='1993-01-29',   # SPY inception date
                  end='2024-12-31',
                  progress=False)

print(f"Downloaded {len(spy)} trading days")
print(f"Date range: {spy.index[0].date()} to {spy.index[-1].date()}")
print(f"\nColumns: {list(spy.columns)}")
print(f"\nFirst 5 rows:")
print(spy.head())
print(f"\nMissing values: {spy.isnull().sum().sum()}")

# ── Calculate log returns ─────────────────────────────────────
spy['Log_Return'] = np.log(spy['Close'] / spy['Close'].shift(1))
spy = spy.dropna()

print(f"\nLog return statistics:")
print(spy['Log_Return'].describe())

# ── Save to CSV ───────────────────────────────────────────────
spy.to_csv('data/spy_data.csv')
print("\nData saved to data/spy_data.csv")

# ── Quick sanity plot ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))

ax1.plot(spy.index, spy['Close'], color='#2166ac', linewidth=0.8)
ax1.set_title('SPY Closing Price (1993–2024)', fontweight='bold', fontsize=13)
ax1.set_ylabel('Price (USD)')
ax1.grid(alpha=0.3)

ax2.plot(spy.index, spy['Log_Return'], color='#d6604d',
         linewidth=0.5, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.5, linestyle='--')
ax2.set_title('SPY Daily Log Returns — Volatility Clustering Visible',
              fontweight='bold', fontsize=13)
ax2.set_ylabel('Log Return')
ax2.set_xlabel('Date')
ax2.grid(alpha=0.3)

# Annotate key events
events = {
    '2000-03-24': '2000\nDot-com peak',
    '2008-09-15': '2008\nFinancial crisis',
    '2020-03-23': '2020\nCOVID crash',
    '2022-01-03': '2022\nRate hikes'
}
for date_str, label in events.items():
    date = pd.Timestamp(date_str)
    if date in spy.index:
        ret = spy.loc[date, 'Log_Return']
        ax2.annotate(label, xy=(date, ret),
                     xytext=(date, ret - 0.04),
                     fontsize=7, color='#762a83',
                     ha='center')

plt.tight_layout()
plt.savefig('outputs/01_spy_price_returns.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved: outputs/01_spy_price_returns.png")